### FOLLOW STEPS : Creation of target index files which we will use for merging in elastic search index

- Have one folder called "inlinks" with the maximum count value for each thread, NOTE : There should be only one file corresponding to each thread 
- Same case for outlinks
- Make sure you update the AUTHOR_NAME TAG with your own name as author before running the code
- Create index files in chunk of 1000s and store them in a folder called final_index, Create this folder and then run the code

In [1]:
import os
import json
import re
import string

from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

In [2]:
# Patterns
DOCNO_PATTERN = r"<DOCNO>(.*?)<\\DOCNO>"
TEXT_PATTERN = r"<TEXT>(.*?)<\\TEXT>"
HEAD_PATTERN = r"<HEAD>(.*?)<\\HEAD>"
DOC_PATTERN = r"<DOC>(.*?)<\\DOC>"


AUTHOR_NAME="Saumya"
INLINKS_FOLDER='./inlinks'
OUTLINKS_FOLDER='./outlinks'
WEBDATA_FOLDER='./webpage db'
FINAL_INDEX_FOLDER='./final_index'

### Merge Inlinks

In [3]:
inlinks_set={}
for inlink_file in os.listdir(INLINKS_FOLDER):
    curr_path=os.path.join(os.getcwd(),f'{INLINKS_FOLDER}/{inlink_file}')
    with open(curr_path,'r') as curr_file:
        inlink_data=json.load(curr_file)
        for url,inlink_list in inlink_data.items():
            if url in inlinks_set:
                inlinks_set[url].append(inlink_list)
            else:
                inlinks_set[url]=inlink_list

with open('./final_inlinks.json','w') as final_inlinks_file:
    json.dump(inlinks_set,final_inlinks_file,indent=2)
    print("Process complete")

Process complete


### Merge Outlinks

In [4]:
outlinks_set={}
for outlink_file in os.listdir(OUTLINKS_FOLDER):
    curr_path=os.path.join(os.getcwd(),f'{OUTLINKS_FOLDER}/{outlink_file}')
    with open(curr_path,'r') as curr_file:
        outlink_data=json.load(curr_file)
        for url,outlink_list in outlink_data.items():
            if url in outlinks_set:
                outlinks_set[url].append(outlink_list)
            else:
                outlinks_set[url]=outlink_list

with open('./final_outlinks.json','w') as final_outlinks_file:
    json.dump(outlinks_set,final_outlinks_file,indent=2)
    print("Process complete")

Process complete


## Preprocessing function

In [5]:
swPath = "./stoplist.txt"

with open(swPath) as file:
    stopwords = file.readlines()
    
    for index, stopword in enumerate(stopwords):
        stopwords[index] = stopword.split("\n")[0]
        

# Adding punctuations in the stopwords list
punctuations = list(string.punctuation)

# Extra
extraPunc = ["``", "'s'", "'", "''"]
[punctuations.append(el) for el in extraPunc]

for p in punctuations:
    stopwords.append(p)
        
print(f'Total number of stopwords: {len(stopwords)}')

Total number of stopwords: 531


In [6]:
ps = PorterStemmer()

def stemWord(word):
    return ps.stem(word)

def inStopWords(word):
    return word.lower() in stopwords

In [7]:
# Function for checking if a word is a number 
def number(word):
    if word[0].isdigit() or word.isdigit():
        return True
    
    try:
        float_value = float(word)
        return True
    except ValueError:
        return False
    
# Helper method for processing text
def processText(text):
    words = word_tokenize(text)
    processedText = []
    
    for index, word in enumerate(words):
        if not inStopWords(word) and not number(word):
            processedText.append(stemWord(word))
            
    return " ".join(processedText)

### INDEX CREATION : Make sure to follow the markdown at the start of the page

In [8]:
# As we have already stored 1000 docs in each file, we can create a final index for each file directly, feel free to adjust the logic if needed in your case if the total file count is different
file_id=1
empty_inlinks=0
empty_outlinks=0


webdata_files = os.listdir(WEBDATA_FOLDER)

print(f'Total files: {len(webdata_files)}')

for index, webdata_file in enumerate(webdata_files):
    print(f'On index: {index}')
    
    index_data={}
    curr_web_file=os.path.join(os.getcwd(),f'{WEBDATA_FOLDER}/{webdata_file}')
    with open(curr_web_file,'r',encoding='utf-8') as curr_file:
        curr_web_data=curr_file.read()
        documents=re.findall(DOC_PATTERN,curr_web_data,re.DOTALL)
        for document in documents:
            curr_index_info={}
            doc_url=re.findall(DOCNO_PATTERN,document,re.DOTALL)[0]
            doc_text=re.findall(TEXT_PATTERN,document,re.DOTALL)
            doc_text='\n'.join(doc_text)
            head_text=re.findall(HEAD_PATTERN,document,re.DOTALL)
            head_text='\n'.join(head_text)
            curr_index_info["content"]=processText(doc_text)
            curr_index_info["title"]=head_text
            curr_index_info["inlinks"]=inlinks_set.get(doc_url,[])
            curr_index_info["outlinks"]=outlinks_set.get(doc_url,[])
            if(len(curr_index_info["inlinks"])==0):
                empty_inlinks+=1
            if(len(curr_index_info["outlinks"])==0):
                empty_outlinks+=1
            curr_index_info["author"] = []
            curr_index_info["author"].append(AUTHOR_NAME)
            index_data[doc_url]=curr_index_info
        with open(f'{FINAL_INDEX_FOLDER}/index_file_{file_id}.json','w') as index_file:
            json.dump(index_data,index_file,indent=2)
        file_id+=1
print(f"Total empty inlinks : {empty_inlinks}")
print(f"Total empty outlinks : {empty_outlinks}")

Total files: 60
On index: 0
On index: 1
On index: 2
On index: 3
On index: 4
On index: 5
On index: 6
On index: 7
On index: 8
On index: 9
On index: 10
On index: 11
On index: 12
On index: 13
On index: 14
On index: 15
On index: 16
On index: 17
On index: 18
On index: 19
On index: 20
On index: 21
On index: 22
On index: 23
On index: 24
On index: 25
On index: 26
On index: 27
On index: 28
On index: 29
On index: 30
On index: 31
On index: 32
On index: 33
On index: 34
On index: 35
On index: 36
On index: 37
On index: 38
On index: 39
On index: 40
On index: 41
On index: 42
On index: 43
On index: 44
On index: 45
On index: 46
On index: 47
On index: 48
On index: 49
On index: 50
On index: 51
On index: 52
On index: 53
On index: 54
On index: 55
On index: 56
On index: 57
On index: 58
On index: 59
Total empty inlinks : 0
Total empty outlinks : 541
